Set up parameters so a job can trigger batch updates dynamically:


In [0]:
# Create parameters for workspace catalog and batch number
dbutils.widgets.text("catalog_name", "meridian_t07", "Catalog Name")
dbutils.widgets.text("batch_num", "1", "Batch Number")

catalog = dbutils.widgets.get("catalog_name")
batch_num = dbutils.widgets.get("batch_num")
volume_path = f"/Volumes/{catalog}/landing/raw_files"

In [0]:
%sql
-- 1. Ensure catalog exists and set context
CREATE CATALOG IF NOT EXISTS meridian_t07;
USE CATALOG meridian_t07;

In [0]:
%sql
-- 2. Create the required schemas
CREATE SCHEMA IF NOT EXISTS landing COMMENT 'Raw file drops';
CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Append-only raw tables';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Validated, conformed tables';
CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Business-ready models and metric views';
CREATE SCHEMA IF NOT EXISTS gov COMMENT 'Security functions and entitlement tables';
CREATE SCHEMA IF NOT EXISTS ops COMMENT 'Data quality results and monitoring';

In [0]:
%sql
-- 3. Create the Managed Volume for raw landing files
CREATE VOLUME IF NOT EXISTS meridian_t07.landing.raw_files
COMMENT 'Landing zone for batch file drops';

In [0]:
%sql
SHOW VOLUMES IN meridian_t07.landing;

Extract 1 quarter of data from samples.tpcds_sf1.inventory and write it into sub-folders per batch:

In [0]:
from pyspark.sql import functions as F



# 1. Read date_dim to find date surrogate keys for a single quarter (Q1 2001)
quarter_dates = spark.sql("""
    SELECT d_date_sk 
    FROM samples.tpcds_sf1.date_dim 
    WHERE d_year = 2001 AND d_qoy = 1
""")

# 2. Slice inventory fact table for that quarter only
inventory_df = spark.table("samples.tpcds_sf1.inventory")
inventory_quarter = inventory_df.join(
    quarter_dates, 
    inventory_df.inv_date_sk == quarter_dates.d_date_sk, 
    "inner"
)

# 3. Simulate disjoint batches using modulo
batch_data = inventory_quarter.filter(f"inv_date_sk % 3 == {int(batch_num) - 1}")

# 4. Inject controlled bad data (Null key using PySpark F.lit(None))
bad_row = batch_data.limit(1).withColumn("inv_item_sk", F.lit(None))
batch_data_with_errors = batch_data.union(bad_row)

# 5. Write to the Unity Catalog Managed Volume
output_path = f"{volume_path}/inventory/batch_{batch_num}"
batch_data_with_errors.write.format("json").mode("overwrite").save(output_path)

print(f"Successfully landed Batch {batch_num} to {output_path}")